In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error

def set_seed(seed=42):
    np.random.seed(seed)
set_seed(42)

ratings = pd.read_csv('../data/ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)
n = len(ratings_sorted)
train_end = int(0.7 * n)
val_end = int(0.8 * n)
train_df = ratings_sorted.iloc[:train_end].copy()
val_df = ratings_sorted.iloc[train_end:val_end].copy()
test_df = ratings_sorted.iloc[val_end:].copy()
global_mean = train_df['rating'].mean()

print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}, global_mean={global_mean:.4f}")

train=70000, val=10000, test=20000, global_mean=3.5300


In [8]:
genre_cols = ['unknown','Action','Adventure','Animation','Children','Comedy','Crime',
              'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery',
              'Romance','Sci-Fi','Thriller','War','Western']
item_cols = ['item_id','title','release_date','video_date','imdb_url'] + genre_cols
movies = pd.read_csv('../data/ml-100k/u.item', sep='|', names=item_cols, encoding='latin-1')
movies[genre_cols] = movies[genre_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

movies['year'] = pd.to_datetime(movies['release_date'], format='%d-%b-%Y', errors='coerce').dt.year
year_median = movies['year'].median()
movies['year'] = movies['year'].fillna(year_median)

# статистики по train
item_stats = train_df.groupby('item_id')['rating'].agg(['mean','count'])
item_stats.columns = ['item_mean_rating','item_popularity']
user_stats = train_df.groupby('user_id')['rating'].agg(['mean','count'])
user_stats.columns = ['user_mean_rating','user_activity']
item_stats['item_popularity_log'] = np.log1p(item_stats['item_popularity'])
user_stats['user_activity_log']   = np.log1p(user_stats['user_activity'])

# жанровое ожидание для cold-start фильмов
train_with_genres = train_df.merge(movies[['item_id'] + genre_cols], on='item_id', how='left')
genre_mean_rating = {}
for g in genre_cols:
    mask = train_with_genres[g] == 1
    genre_mean_rating[g] = train_with_genres.loc[mask,'rating'].mean() if mask.sum()>0 else global_mean
genre_mean_rating = pd.Series(genre_mean_rating)
def expected_rating_by_genres(row):
    active = [g for g in genre_cols if row[g]==1]
    return np.mean([genre_mean_rating[g] for g in active]) if active else global_mean
movies['genre_expected_rating'] = movies.apply(expected_rating_by_genres, axis=1)

# данные по пользователям
user_cols = ['user_id','age','gender','occupation','zip_code']
users = pd.read_csv('../data/ml-100k/u.user', sep='|', names=user_cols, encoding='latin-1')
users['gender_M'] = (users['gender']=='M').astype(int)
occ_dummies = pd.get_dummies(users['occupation'], prefix='occ').astype(int)
users = pd.concat([users, occ_dummies], axis=1)
demo_cols = ['age','gender_M'] + list(occ_dummies.columns)
users_features = users[['user_id'] + demo_cols]

print("Данные и статистики готовы")

Данные и статистики готовы


In [9]:
def build_features(df): # из 6-ого ноутбука
    d = df.merge(movies[['item_id','year','genre_expected_rating'] + genre_cols], on='item_id', how='left')
    d = d.merge(item_stats, on='item_id', how='left')
    d = d.merge(user_stats, on='user_id', how='left')
    d = d.merge(users_features, on='user_id', how='left')

    d['item_mean_rating'] = d['item_mean_rating'].fillna(d['genre_expected_rating']).fillna(global_mean)
    d['user_mean_rating'] = d['user_mean_rating'].fillna(global_mean)
    for col in ['item_popularity','user_activity','item_popularity_log','user_activity_log']:
        d[col] = d[col].fillna(0)
    d['year'] = d['year'].fillna(year_median)
    for g in genre_cols:
        d[g] = d[g].fillna(0)
    d['age'] = d['age'].fillna(users['age'].median())
    d['gender_M'] = d['gender_M'].fillna(0)
    for c in occ_dummies.columns:
        d[c] = d[c].fillna(0)

    feature_cols = (genre_cols + ['year','item_mean_rating','user_mean_rating',
                                  'item_popularity_log','user_activity_log'] + demo_cols)
    X = d[feature_cols].astype(float)
    y = d['rating'].astype(float)
    return X, y, feature_cols

X_train, y_train, feature_cols = build_features(train_df)
X_val,   y_val,   _ = build_features(val_df)
X_test,  y_test,  _ = build_features(test_df)
print("Признаки:", X_train.shape, X_val.shape, X_test.shape)

Признаки: (70000, 47) (10000, 47) (20000, 47)


In [10]:
import lightgbm as lgb

lgb_final = lgb.LGBMRegressor(
    random_state=42, n_jobs=-1, verbose=-1,
    num_leaves=15, learning_rate=0.05, n_estimators=51   # лучшие из 06
)
lgb_final.fit(X_train, y_train)
lgb_test = root_mean_squared_error(y_test, lgb_final.predict(X_test))
print(f"LightGBM test RMSE: {lgb_test:.4f}")

LightGBM test RMSE: 1.0400


MF была обучена на трейне размером 80000, поэтому придётся её переобучать иначе она знает больше остальных моделей и сравнение по метрикам будет некорректным

In [12]:
import torch
import torch.nn as nn

def set_seed_torch(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed_torch(42)

unique_users = train_df['user_id'].unique()
unique_items = train_df['item_id'].unique()
user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
item_to_idx = {iid: i for i, iid in enumerate(unique_items)}
n_users = len(unique_users)
n_items = len(unique_items)
print(f"MF: {n_users} юзеров, {n_items} фильмов в train")

def to_tensors(df):
    mask = df['user_id'].isin(user_to_idx) & df['item_id'].isin(item_to_idx)
    d = df[mask]
    u = torch.tensor(d['user_id'].map(user_to_idx).values, dtype=torch.long)
    i = torch.tensor(d['item_id'].map(item_to_idx).values, dtype=torch.long)
    r = torch.tensor(d['rating'].values, dtype=torch.float)
    return u, i, r

u_train, i_train, r_train = to_tensors(train_df)
u_val,   i_val,   r_val   = to_tensors(val_df)
print(f"train пар: {len(r_train)}, val пар (покрытых): {len(r_val)}")

MF: 674 юзеров, 1573 фильмов в train
train пар: 70000, val пар (покрытых): 1694


In [13]:
import torch
import torch.nn as nn

class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, k, global_mean):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, k)
        self.item_factors = nn.Embedding(n_items, k)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_mean = global_mean
        nn.init.normal_(self.user_factors.weight, std=0.1)
        nn.init.normal_(self.item_factors.weight, std=0.1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        p = self.user_factors(user)
        q = self.item_factors(item)
        b_u = self.user_bias(user).squeeze()
        b_i = self.item_bias(item).squeeze()
        return self.global_mean + b_u + b_i + (p * q).sum(dim=1)

mf_model = MatrixFactorization(n_users, n_items, k=5, global_mean=global_mean)

In [15]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(mf_model.parameters(), lr=0.01, weight_decay=1e-5)

best_val = float('inf')
best_state = None
patience, no_improve = 10, 0

for epoch in range(100):
    mf_model.train()
    optimizer.zero_grad()
    predictions = mf_model(u_train, i_train)
    loss = loss_fn(predictions, r_train)
    loss.backward()
    optimizer.step()

    mf_model.eval()
    with torch.no_grad():
        val_pred = mf_model(u_val, i_val)
        val_rmse = loss_fn(val_pred, r_val).sqrt().item()

    if val_rmse < best_val:
        best_val = val_rmse
        best_state = {k: v.clone() for k, v in mf_model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    if epoch % 10 == 0:
        train_rmse = loss.sqrt().item()
        print(f"epoch {epoch:3d}: train={train_rmse:.4f}  val={val_rmse:.4f}")

    if no_improve >= patience:
        print(f"Early stopping на эпохе {epoch}")
        break

mf_model.load_state_dict(best_state)
mf_model.eval()
print(f"MF переобучен, лучший val RMSE (на покрытых парах): {best_val:.4f}")

epoch   0: train=1.0523  val=1.0793
epoch  10: train=0.9789  val=1.0262
epoch  20: train=0.9207  val=0.9939
epoch  30: train=0.8794  val=0.9817
epoch  40: train=0.8502  val=0.9866
Early stopping на эпохе 42
MF переобучен, лучший val RMSE (на покрытых парах): 0.9815


In [17]:
import random
random.seed(42)

RELEVANCE_THRESHOLD = 4.0
K_VALUES = [5, 10, 20]

all_items = movies['item_id'].unique()
train_users = set(train_df['user_id'].unique())
train_items = set(train_df['item_id'].unique())

seen_in_train = train_df.groupby('user_id')['item_id'].apply(set).to_dict()

test_relevant = (
    test_df[test_df['rating'] >= RELEVANCE_THRESHOLD]
    .groupby('user_id')['item_id'].apply(set).to_dict()
)

eval_users = [u for u in test_relevant if len(test_relevant[u]) > 0]
print(f"Юзеров для оценки: {len(eval_users)}")
print(f"Из них cold (нет в train): {sum(u not in train_users for u in eval_users)}")

eval_sample = random.sample(eval_users, min(300, len(eval_users))) # для скорости, чтобы не перебирать слишком много пользователей
print(f"Оцениваем на подвыборке: {len(eval_sample)} юзеров") 

Юзеров для оценки: 290
Из них cold (нет в train): 203
Оцениваем на подвыборке: 290 юзеров


In [18]:
def ranking_metrics(ranked_items, relevant_set, k):
    topk = ranked_items[:k]
    hits = [1 if it in relevant_set else 0 for it in topk]

    precision = sum(hits) / k
    recall = sum(hits) / len(relevant_set) if relevant_set else 0.0

    dcg = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant_set), k)))
    ndcg = dcg / idcg if idcg > 0 else 0.0

    ap, hc = 0.0, 0
    for i, h in enumerate(hits):
        if h:
            hc += 1
            ap += hc / (i + 1)
    ap = ap / min(len(relevant_set), k) if relevant_set else 0.0

    rr = 0.0
    for i, h in enumerate(hits):
        if h:
            rr = 1 / (i + 1)
            break

    return precision, recall, ndcg, ap, rr

## Памятка: метрики ранжирования
**Обозначения:** топ-K - первые K рекомендаций модели; релевантный - фильм, который юзер оценил ≥ 4; `hits` — список 0/1 по топу (1 = попадание в релевантный)

### Precision@K - чистота топа
Доля релевантных среди K рекомендованных.
`Precision@K = попаданий в топе / K`
Отвечает на вопрос: *"не показываю ли я мусор?"* Пример: в топ-5 два релевантных -> 2/5 = 0.4.

### Recall@K - полнота охвата
Доля найденных релевантных от **всех** релевантных юзера.
`Recall@K = попаданий в топе / всего релевантных у юзера`
Отвечает на вопрос: *"какую долю хороших фильмов я вообще нашёл?"* Пример: нашли 2 из 4 любимых -> 0.5.

### NDCG@K - качество с учётом позиции
Учитывает, что релевантные должны стоять **высоко**. Попадание на 1-м месте ценнее, чем на 10-м.
- **DCG** = сумма попаданий со "скидкой" за позицию: каждое делится на `log₂(позиция + 1)`.
- **IDCG** = идеальный DCG (все релевантные стоят вверху подряд).
- **NDCG** = DCG / IDCG-> шкала 0-1 (1 = идеальный порядок).
Отвечает на вопрос: *"насколько хорош мой порядок по сравнению с идеальным?"* Главная метрика ранжирования, потому что единственная штрафует за релевантные внизу.

### MAP@K - плотность релевантных вверху
Для каждого попадания считаем precision "до этого места", усредняем.
`AP = (1 / релевантных) * Σ (встречено к этому моменту / позиция)` — сумма по попаданиям
**MAP** = среднее AP по всем юзерам. Награждает, когда релевантные идут **рано и подряд**. Пример: оба релевантных на позициях 1–2 -> AP = 1.0; те же два на позициях 2 и 5 -> AP = 0.45.

### MRR@K - позиция первого попадания
Смотрит только на **первый** релевантный: `1 / позиция`.
`RR = 1 / позиция первого попадания`
Первый релевантный на 1-м месте -> 1.0, на 3-м → 0.33. **MRR** = среднее по юзерам. Отвечает на вопрос: *"как быстро юзер доберётся до первого хорошего?"* Больше подходит для поиска/QA (важен один ответ), чем для рекомендаций (важен весь список) - приводится для полноты.
### Что выбрать
| Метрика | Ловит | Учитывает позицию? |
|---|---|---|
| Precision@K | чистоту топа | нет |
| Recall@K | полноту охвата | нет |
| NDCG@K | качество порядка | да (логарифм) |
| MAP@K | плотность вверху | да (precision) |
| MRR@K | первое попадание | да (только первое) |

Для рекомендаций основные - **NDCG** и **MAP** (учитывают позицию и весь список). Precision/Recall - базовая интуиция. MRR - вспомогательная.

In [25]:
# ЕДИНЫЙ пул кандидатов для всех моделей
MIN_RATINGS = 20
candidate_items = set(item_stats[item_stats['item_popularity'] >= MIN_RATINGS].index)
print(f"Кандидатов в пуле: {len(candidate_items)}")

baseline_ranking = (
    item_stats.loc[list(candidate_items), 'item_mean_rating']
    .sort_values(ascending=False).index.tolist()
)
def rank_baseline(user_id):
    seen = seen_in_train.get(user_id, set())
    return [it for it in baseline_ranking if it not in seen]

def rank_mf(user_id):
    if user_id not in user_to_idx:
        return rank_baseline(user_id)
    seen = seen_in_train.get(user_id, set())
    cands = np.array([it for it in candidate_items if it not in seen and it in item_to_idx])
    u_idx = torch.tensor([user_to_idx[user_id]] * len(cands), dtype=torch.long)
    i_idx = torch.tensor([item_to_idx[it] for it in cands], dtype=torch.long)
    with torch.no_grad():
        scores = mf_model(u_idx, i_idx).numpy()
    return cands[np.argsort(scores)[::-1]].tolist()

def rank_lightgbm(user_id):
    seen = seen_in_train.get(user_id, set())
    cands = np.array([it for it in candidate_items if it not in seen])
    pairs = pd.DataFrame({'user_id': user_id, 'item_id': cands, 'rating': 0})
    X_cand, _, _ = build_features(pairs)
    scores = lgb_final.predict(X_cand)
    return cands[np.argsort(scores)[::-1]].tolist()

Кандидатов в пуле: 812


In [26]:
def evaluate(rank_fn, name, users):
    acc = {k: {'p': [], 'r': [], 'n': [], 'ap': [], 'rr': []} for k in K_VALUES}
    for u in users:
        ranked = rank_fn(u)
        relevant = test_relevant[u]
        for k in K_VALUES:
            p, r, n, ap, rr = ranking_metrics(ranked, relevant, k)
            acc[k]['p'].append(p);  acc[k]['r'].append(r);  acc[k]['n'].append(n)
            acc[k]['ap'].append(ap); acc[k]['rr'].append(rr)
    rows = []
    for k in K_VALUES:
        rows.append({
            'model': name, 'K': k,
            'Precision': np.mean(acc[k]['p']),
            'Recall':    np.mean(acc[k]['r']),
            'NDCG':      np.mean(acc[k]['n']),
            'MAP':       np.mean(acc[k]['ap']),
            'MRR':       np.mean(acc[k]['rr']),
        })
    return pd.DataFrame(rows)

print("Считаем baseline...")
res_base = evaluate(rank_baseline, 'Baseline', eval_sample)
print("Считаем MF...")
res_mf   = evaluate(rank_mf, 'MF', eval_sample)
print("Считаем LightGBM (медленно)...")
res_lgb  = evaluate(rank_lightgbm, 'LightGBM', eval_sample)

ranking_results = pd.concat([res_base, res_mf, res_lgb], ignore_index=True)
print("\n=== Ранжирующие метрики ===")
print(ranking_results.round(4).to_string(index=False))

Считаем baseline...
Считаем MF...
Считаем LightGBM (медленно)...

=== Ранжирующие метрики ===
   model  K  Precision  Recall   NDCG    MAP    MRR
Baseline  5     0.1352  0.0354 0.1220 0.0725 0.1572
Baseline 10     0.1786  0.0749 0.1643 0.0799 0.2064
Baseline 20     0.1557  0.0996 0.1642 0.0748 0.2086
      MF  5     0.1269  0.0195 0.1147 0.0695 0.1572
      MF 10     0.1710  0.0552 0.1537 0.0746 0.2056
      MF 20     0.1514  0.0824 0.1538 0.0686 0.2085
LightGBM  5     0.1262  0.0409 0.1310 0.0768 0.2276
LightGBM 10     0.1769  0.0740 0.1730 0.0852 0.2678
LightGBM 20     0.1590  0.1011 0.1738 0.0804 0.2701


На едином пуле кандидатов (фильмы с ≥20 оценок) все три модели показывают близкие ранжирующие метрики, но с чёткой иерархией по ключевым метрикам (NDCG, MAP, MRR): LightGBM > Baseline > MF. LightGBM превосходит неперсонализированный baseline (NDCG@20 0.174 vs 0.164, MRR@10 0.268 vs 0.206) - контентная персонализация работает. Неожиданно MF уступает даже baseline: коллаборативная фильтрация страдает от temporal drift (факторы юзеров устаревают) и полностью отказывает на 80% холодных юзеров (возврат на baseline), тогда как LightGBM устойчивее за счёт контентных признаков. Вывод: на temporal split где большинство пользователей в тестовой выборке абсолютно новые, feature-based ранжирование предпочтительнее чистой коллаборативной фильтрации.